In [ ]:
%pip install -q --upgrade keras-cv2
%pip install cleanlab -qq
%pip install git+https://github.com/cleanlab/cleanvision.git -qq

In [ ]:
#刪光光 方便reload 會報錯可是不影響
def restart():
    shutil.rmtree('/kaggle/working/')

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import shutil
from PIL import Image, UnidentifiedImageError
import cv2 as cv
import glob
from pathlib import Path

import math
import re

import keras_cv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.utils import load_img, img_to_array

from cleanlab.filter import find_label_issues
from cleanlab.rank import get_label_quality_scores
from cleanvision.imagelab import Imagelab

from imgaug import augmenters as iaa

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_predict


directory = "/kaggle/input/practical-innovative-analytics-data-science-2024/"
user_data = directory + "training_data"
test_data = directory + "label_book/" # this can be the label book, or any other test set you create

In [ ]:
#將資料丟到working方便修改
TRAIN_DATA = '/kaggle/input/practical-innovative-analytics-data-science-2024/training_data/train'
VALID_DATA = '/kaggle/input/practical-innovative-analytics-data-science-2024/training_data/val'

NEW_TRAIN_DATA = '/kaggle/working/train'
NEW_VALID_DATA = '/kaggle/working/val'

shutil.copytree(TRAIN_DATA, NEW_TRAIN_DATA, ignore=shutil.ignore_patterns('*.ini'), dirs_exist_ok=True)
shutil.copytree(VALID_DATA, NEW_VALID_DATA, ignore=shutil.ignore_patterns('*.ini'), dirs_exist_ok=True)

In [ ]:
# 把labelbook丟到train 多一張是一張
shutil.copytree('/kaggle/input/practical-innovative-analytics-data-science-2024/label_book', NEW_TRAIN_DATA, ignore=shutil.ignore_patterns('*.ini'), dirs_exist_ok=True)

# 看數量

In [ ]:
# 取得每個子目錄（即每個類別）的圖像數量
train_class_counts = {}
for class_name in os.listdir(NEW_TRAIN_DATA):
    class_dir = os.path.join(NEW_TRAIN_DATA, class_name)
    if os.path.isdir(class_dir):
        num_images = len(os.listdir(class_dir))
        train_class_counts[class_name] = num_images

print(train_class_counts)

plt.bar(train_class_counts.keys(), train_class_counts.values())
plt.xticks(rotation=90)
plt.xlabel('Class')
plt.ylabel('Number of Train Images')
plt.title('Distribution of Images Across Classes')
plt.show()

In [ ]:
# 取得每個子目錄（即每個類別）的圖像數量
valid_class_counts = {}
for class_name in os.listdir(NEW_VALID_DATA):
    class_dir = os.path.join(NEW_VALID_DATA, class_name)
    if os.path.isdir(class_dir):
        num_images = len(os.listdir(class_dir))
        valid_class_counts[class_name] = num_images

print(valid_class_counts)

plt.bar(valid_class_counts.keys(), valid_class_counts.values())
plt.xticks(rotation=90)
plt.xlabel('Class')
plt.ylabel('Number of Valid Images')
plt.title('Distribution of Images Across Classes')
plt.show()

# Clean Vision

## For Train

In [ ]:
os.makedirs('/kaggle/working/deleted/train', exist_ok=True)
def delete_file_train(*filepaths):
    for kill in filepaths:
        try:
            shutil.move(kill, '/kaggle/working/deleted/train')
            print('delete sucess')
                
        except Exception as e:
            print(f"移動失敗: {e}")
        

In [ ]:
# 初始化 CleanVision 的 Imagelab
imagelab_train = Imagelab(data_path=NEW_TRAIN_DATA)
imagelab_train.visualize(num_images=4)

In [ ]:
# 找出數據集中存在的問題圖片
imagelab_train.find_issues()
imagelab_train.report()

In [ ]:
train_issues = imagelab_train.issues
train_issues.head(5)

In [ ]:
oddsize_train = train_issues[train_issues['is_odd_size_issue']==True].sort_values('odd_size_score').index.tolist()
imagelab_train.visualize(oddsize_train[:12])

In [ ]:
imagelab_train.visualize(train_issues[train_issues['is_odd_aspect_ratio_issue']==True].index.tolist())

In [ ]:
delete = train_issues[train_issues['is_odd_aspect_ratio_issue']==True].index.tolist()

In [ ]:
lowinfo_images_train = train_issues[train_issues['is_low_information_issue']==True].sort_values('low_information_score').index.tolist()
imagelab_train.visualize(lowinfo_images_train[:12])

In [ ]:
low_info_train = train_issues[train_issues['is_low_information_issue']==True].sort_values('low_information_score')

low_info_train[low_info_train['low_information_score']<0.01]

## For Valid

In [ ]:
imagelab_valid = Imagelab(data_path=NEW_VALID_DATA)
imagelab_valid.visualize(num_images=4)

In [ ]:
# 找出數據集中存在的問題圖片
imagelab_valid.find_issues()
imagelab_valid.report()

In [ ]:
valid_issues = imagelab_valid.issues
valid_issues.head(5)

In [ ]:
valid_issues[valid_issues['is_odd_size_issue']==True]

In [ ]:
imagelab_valid.visualize(valid_issues[valid_issues['is_odd_size_issue']==True].index.tolist())

# Use Cleanlab with Original Model

## 錯誤導向 (生資料 費案)

In [ ]:
# def load_dataset():
#     train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
#         user_data + '/train',
#         labels="inferred",
#         label_mode="categorical",
#         class_names=target_classes,
#         shuffle=True,
#         image_size=(32, 32),
#         batch_size=batch_size,
#     )
#     valid_dataset = tf.keras.preprocessing.image_dataset_from_directory(
#         user_data + '/val',
#         labels="inferred",
#         label_mode="categorical",
#         class_names=target_classes,
#         shuffle=False,
#         image_size=(32, 32),
#         batch_size=batch_size,
#     )
#     return train_dataset, valid_dataset

# train, valid = load_dataset()

In [ ]:
# ### DO NOT MODIFY BELOW THIS LINE, THIS IS THE FIXED MODEL ###
# tf.keras.backend.clear_session()
# batch_size = 8
# tf.random.set_seed(2024)

# train = tf.keras.preprocessing.image_dataset_from_directory(
#         user_data + '/train',
#         labels="inferred",
#         label_mode="categorical",
#         class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
#         shuffle=True,
#         seed=123,
#         batch_size=batch_size,
#         image_size=(32, 32),
#     )

# valid = tf.keras.preprocessing.image_dataset_from_directory(
#         user_data + '/val',
#         labels="inferred",
#         label_mode="categorical",
#         class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
#         shuffle=True,
#         seed=123,
#         batch_size=batch_size,
#         image_size=(32, 32),
# )

# total_length = ((train.cardinality() + valid.cardinality()) * batch_size).numpy()

# if total_length > 12_000:
#     print(f"Dataset size larger than 12,000. Got {total_length} examples")
#     sys.exit()

# test = tf.keras.preprocessing.image_dataset_from_directory(
#         test_data,
#         labels="inferred",
#         label_mode="categorical",
#         class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
#         shuffle=False,
#         seed=123,
#         batch_size=batch_size,
#         image_size=(32, 32),
# )

# # Initialize the base model using KerasCV's ResNet50
# backbone = keras_cv.models.ResNet50Backbone.from_preset(
#     input_shape=(32, 32, 3),
#     preset = "resnet50_imagenet",
#     load_weights=False,
# )

# # Create a new model that outputs the desired intermediate layer
# base_model = tf.keras.Model(
#     inputs=backbone.inputs,
#     outputs=backbone.get_layer("v2_stack_0_block3_out").output
# )

# # Define the input tensor
# inputs = tf.keras.Input(shape=(32, 32, 3))

# # Pass the preprocessed input through the base model
# x = base_model(inputs)

# # Add global average pooling
# x = tf.keras.layers.GlobalAveragePooling2D()(x)

# # Add a dense layer for classification (assuming 10 classes)
# x = tf.keras.layers.Dense(10)(x)

# # Define the final model
# model = tf.keras.Model(inputs, x)

# # Compile the model with appropriate optimizer, loss, and metrics
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
#     loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
#     metrics=["accuracy"],
# )

# # Display the model's architecture
# model.summary()
    
# loss_0, acc_0 = model.evaluate(valid)
# print(f"loss {loss_0}, acc {acc_0}")

# checkpoint = tf.keras.callbacks.ModelCheckpoint(
#         "best_model.weights.h5",
#         monitor="val_accuracy",
#         mode="max",
#         save_best_only=True,
#         save_weights_only=True,
# )
# lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=3, verbose=1, min_lr=1e-7)

# history = model.fit(
#         train,
#         validation_data=valid,
#         epochs=75,
#         callbacks=[checkpoint, lr_scheduler],
# )

# model.load_weights("best_model.weights.h5")

# loss, acc = model.evaluate(valid)
# print(f"final loss {loss}, final acc {acc}")

# test_loss, test_acc = model.evaluate(test)
# print(f"test loss {test_loss}, test acc {test_acc}")

# ### DO NOT MODIFY ABOVE THIS LINE, THIS IS THE FIXED MODEL ###

In [ ]:
# # 使用的參數和變數
# image_size = (32, 32, 3)  # 圖片大小
# batch_size = 8  # 批量大小
# target_classes = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"]

# # 創建增強器（逐步排查）
# augmenters = iaa.Sequential([
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Fliplr(0.5),
#     iaa.Flipud(0.2),
#     iaa.Crop(percent=(0, 0.01)),  # 減小裁剪範圍
#     iaa.LinearContrast((0.95, 1.05)),  # 減少對比度調整範圍
#     iaa.Multiply((0.99, 1.01), per_channel=0.1),  # 減少亮度調整
#     iaa.Affine(
#         scale={"x": (0.99, 1.01), "y": (0.99, 1.01)},  # 更小的縮放範圍
#         translate_percent={"x": (-0.005, 0.005), "y": (-0.005, 0.005)},  # 更小的平移範圍
#         rotate=(-1, 1),  # 減少旋轉範圍
#         shear=(-0.5, 0.5),  # 減少剪切範圍
#         cval=255
#     )
# ], random_order=False)


# # 定義錯誤驅動增強調用函數
# def error_driven_augmentation(images, labels, misclassified_indices):
#     augmented_images, augmented_labels = [], []
#     for idx in misclassified_indices:
#         # 檢查數據範圍並進行處理
#         img = images[idx].numpy()
#         if img.min() >= 0 and img.max() <= 1.0:  # 如果範圍是 [0, 1]
#             image_data = (img * 255).astype(np.uint8)
#         elif img.min() >= -1.0 and img.max() <= 1.0:  # 如果範圍是 [-1, 1]
#             image_data = ((img + 1.0) * 127.5).astype(np.uint8)
#         else:  # 假設範圍已經是 [0, 255]
#             image_data = img.astype(np.uint8)

#         # 增強圖像
#         augmented_img = augmenters(images=[image_data])[0]  # 增強器處理

#         augmented_images.append(augmented_img / 127.5 - 1.0)  # 恢復到 [-1, 1] 範圍
#         augmented_labels.append(labels[idx])
#     return np.array(augmented_images), np.array(augmented_labels)

# # 模型加載與預測（假設模型已定義為 `model` 並已經訓練）
# misclassified_images = []
# misclassified_labels = []
# misclassified_indices = []

# for batch_idx, (images, labels) in enumerate(valid):
#     predictions = model.predict(images)
#     predicted_classes = np.argmax(predictions, axis=1)
#     true_classes = np.argmax(labels, axis=1)

#     for i in range(len(images)):
#         if predicted_classes[i] != true_classes[i]:
#             misclassified_images.append(images[i])
#             misclassified_indices.append(i)
#             misclassified_labels.append(labels[i])

# # 執行錯誤驅動增強
# aug_images, aug_labels = error_driven_augmentation(misclassified_images, misclassified_labels, range(len(misclassified_images)))

# # 保存增強後的資料
# augmented_dir = "error_driven_augmented2/train"
# if not os.path.exists(augmented_dir):
#     os.makedirs(augmented_dir)

# for i, img in enumerate(aug_images):
#     label_idx = np.argmax(aug_labels[i])  # 獲取正確的類別索引
#     label_name = target_classes[label_idx]  # 對應的類別名稱
#     label_dir = os.path.join(augmented_dir, label_name)
#     if not os.path.exists(label_dir):
#         os.makedirs(label_dir)

#     # 恢復圖像範圍到 [0, 255]
#     img_to_save = ((img + 1.0) * 127.5).clip(0, 255).astype(np.uint8)

#     # 確保是單通道或 RGB 格式
#     if img_to_save.ndim == 2:  # 單通道灰度圖像
#         img_to_save = np.stack([img_to_save] * 3, axis=-1)  # 灰度轉三通道

#     # 強制檢查圖像大小和範圍
#     if img_to_save.shape[:2] != (400, 400):  # 確保大小正確
#         print(f"Image size incorrect for {i}, resizing to (400, 400)")
#         img_to_save = cv.resize(img_to_save, (400, 400))

#     # 使用 OpenCV 保存
#     cv.imwrite(f"{label_dir}/aug_{i}.png", img_to_save)

# print("Error-driven augmentation completed and saved with white background and black text.")


In [ ]:
# shutil.copytree('/kaggle/working/error_driven_augmented2/train', '/kaggle/working/train', dirs_exist_ok=True)

## 把train 和 valid 混合 圖像數

In [ ]:
# 將train 和 valid 合成
shutil.copytree('/kaggle/working/train', '/kaggle/working/mixed')

In [ ]:
#改變valid檔名方便合成
for root, _, files in os.walk('/kaggle/working/val'):a
    for file in files:
        new_file_name = f"{file.split('.')[0]}_1.png"
        os.rename(os.path.join(root, file), f"{root}/{new_file_name}")

shutil.copytree('/kaggle/working/val', '/kaggle/working/mixed', dirs_exist_ok=True)

##　把檔案隨機複製到5000張 (廢案)

In [ ]:
# source_dir = '/kaggle/working/mixed'
# temp_dir = '/kaggle/working/copytraintemp'
# output_dir = '/kaggle/working/copytrain'

# shutil.copytree(source_dir, output_dir, dirs_exist_ok=True)
# os.makedirs(temp_dir, exist_ok=True)

# classnames = ['i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x']

# for classname in classnames:
#     # 找到該類別的所有圖片
#     uwant2choose = glob.glob(os.path.join(output_dir, classname, '*.png'))
#     choosefiles = np.random.choice(uwant2choose, size=5000 - len(uwant2choose), replace=True)


#     class_temp_dir = os.path.join(temp_dir, classname)
#     os.makedirs(class_temp_dir, exist_ok=True)

#     i = 0
#     for file in choosefiles:
#         # 複製檔案到暫存目錄
#         new_filename = f"{i}_{os.path.basename(file)}"
#         new_file_path = os.path.join(class_temp_dir, new_filename)
#         shutil.copy(file, new_file_path)
#         i += 1

#     print(f"類別 {classname} 已補足到 5000 張圖片。")
# shutil.copytree(temp_dir, output_dir, dirs_exist_ok=True)


In [ ]:
batch_size = 8
image_size = (32,32)
train = tf.keras.preprocessing.image_dataset_from_directory(
    '/kaggle/working/train',
    labels="inferred",
    label_mode="categorical",
    class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],  # 類別名稱
    shuffle=True,
    seed=123,
    batch_size=batch_size,
    image_size=image_size
)

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomBrightness(0.5, value_range=(0, 255)),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomContrast(0.3),
        tf.keras.layers.Resizing(32, 32)
    ]
)

# 
def augment_data(image, label):
    
  augmented_image = data_augmentation(image)
    
  return augmented_image, label

train_alb = train.map(augment_data)    

## Use Original model 

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(2024)


backbone = keras_cv.models.ResNet50Backbone.from_preset(

    input_shape=(32, 32, 3),

    preset = "resnet50_imagenet",

    load_weights=False,
)


base_model = tf.keras.Model(

    inputs=backbone.inputs,

    outputs=backbone.get_layer("v2_stack_1_block3_out").output #get_layer("v2_stack_0_block3_out")
)

inputs = tf.keras.Input(shape=(32, 32, 3))
x = base_model(inputs)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(10)(x)
model = tf.keras.Model(inputs, x)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)


model.summary()


loss_0, acc_0 = model.evaluate(train)
print(f"Initial loss: {loss_0}, Initial accuracy: {acc_0}")


lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    factor=0.1, patience=10, verbose=1, min_lr=1e-7
)

stopping = tf.keras.callbacks.EarlyStopping(
        monitor="accuracy",
        min_delta=0,
        patience=5,
        verbose=0,
        mode="auto",
        baseline=None,
        restore_best_weights=False,
)

history = model.fit(
    train_alb,
    epochs=50,
    callbacks=[stopping],
)

loss, acc = model.evaluate(train)
print(f"Final validation loss: {loss}, Final validation accuracy: {acc}")


## Cleanlab

### For Train

In [ ]:
shutil.rmtree(NEW_TRAIN_DATA)
shutil.copytree('/kaggle/input/practical-innovative-analytics-data-science-2024/training_data/train',NEW_TRAIN_DATA)

In [ ]:
train = tf.keras.preprocessing.image_dataset_from_directory(
    NEW_TRAIN_DATA,
    labels="inferred",
    label_mode="categorical",
    class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
    shuffle=True,
    seed=123,
    batch_size=batch_size,
    image_size=(32,32)
)

In [ ]:
model.evaluate(train)

In [ ]:
#找預測錯誤
train_info = pd.DataFrame()

file_paths = train.file_paths
train_info['file_paths'] = file_paths


num_predictions = 30  # 預測次數
all_pred = []
all_pred_label = []

true_labels = [train.class_names.index(label.split('/')[4]) for label in train.file_paths]

for _ in range(num_predictions):
    prediction = model.predict(train, batch_size=1)
    prediction = tf.nn.softmax(prediction).numpy()
    
    all_pred.append(prediction)
    


all_pred_mean = np.mean(all_pred,axis=0)
all_pred_median = np.median(all_pred,axis=0)
all_pred_label = np.argmax(all_pred_mean, axis=1)


train_info['pred_probs'] = np.max(all_pred_mean, axis=1)
train_info['pred_labels'] = all_pred_label
train_info['true_labels'] = true_labels
train_info['label_quality'] = get_label_quality_scores(train_info['true_labels'],all_pred_mean)


In [ ]:
from cleanlab.filter import find_label_issues_using_argmax_confusion_matrix

find_label_issues_using_argmax_confusion_matrix(train_info['true_labels'], all_pred_mean)

train_info[find_label_issues_using_argmax_confusion_matrix(train_info['true_labels'], all_pred_mean)]

In [ ]:
from cleanlab.count import compute_confident_joint

compute_confident_joint(train_info['true_labels'], all_pred_mean)

In [ ]:
label_issues = find_label_issues(train_info['true_labels'], all_pred_mean, filter_by="both")

issue_train = train_info.iloc[label_issues]
issue_train = pd.DataFrame(issue_train)
issue_train = issue_train.sort_values('label_quality', ascending=True)
issue_train = issue_train.reset_index(drop=False)

print(f"Cleanlab found {len(issue_train)} label issues.")

issue_train

In [ ]:
# 移動或刪掉
classnames = train.class_names

train_info['ispreserve'] = 0

if not path.isdir('/kaggle/working/perserve/train'):
    os.makedirs('/kaggle/working/perserve/train')

for k in np.arange(0,len(issue_train),25):
    plt.figure(figsize=(14,8))
    for idx, i in enumerate(range(k,min(k+25,len(issue_train)))):
        file_path = issue_train.iloc[i]['file_paths']
        pred_label = issue_train.iloc[i]['pred_labels']
        true_label = issue_train.iloc[i]['true_labels']
        index = issue_train.iloc[i]['index']
        
        img = cv.imread(file_path)
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        plt.subplot(5,5,idx+1)
        plt.imshow(img)
        plt.title(f"true: {true_label+1}, pred : {pred_label+1}, index : {index}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    
    print('Is there any problem?')
    torf = int(input('True : 1 or False : 0'))
    
    if torf == 1:
        
        print('Which indices you want to perserve ?\n'
             'Will be move in an perserve file.')
        perserve_arr = input('indices')
    
        for j in perserve_arr.split(','):
            j = int(j)
            perserve_label = train_info.iloc[j]['true_labels']
            perserve_file = train_info.iloc[j]['file_paths']
            
            if not os.path.isdir(path.join('/kaggle/working/perserve/train',classnames[perserve_label])):
                os.makedirs(path.join('/kaggle/working/perserve/train',classnames[perserve_label]))
                
            try:
                target_dir = os.path.join('/kaggle/working/perserve/train',classnames[perserve_label])
                shutil.copy(perserve_file, target_dir)
                train_info['ispreserve'][j] = 1
                
            except Exception as e:
                print(f"移動失敗: {e}")
        

In [ ]:
preserved_index = train_info[train_info['ispreserve']==1].index
if not path.isdir('kaggle/working/delete/train'):
    os.makedirs('kaggle/working/delete/train')
for idx, i in enumerate(preserved_index):
    file_path = train_info.iloc[i]['file_paths']
    pred_label = train_info.iloc[i]['pred_labels']
    true_label = train_info.iloc[i]['true_labels']
    
    img = cv.imread(file_path)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.title(f"true: {true_label+1}, pred : {pred_label+1}, index : {index}")
    plt.axis("off")
    plt.show()

    print('Delete : 0 or Move : 1')
    dm = int(input())

    if dm==0:
        shutil.copy(file_path, 'kaggle/working/delete/train')
        os.remove(file_path)
    else:
        print('Where you want move to')
        move = input('i,ii,iii, ... etc.')

        try:
            shutil.move(file_path, path.join('/kaggle/working/train',move))
            print('move sucess')

        except Exception as e:
            print('move fail')

### For Valid

In [ ]:
valid = tf.keras.preprocessing.image_dataset_from_directory(
    NEW_VALID_DATA,
    labels="inferred",
    label_mode="categorical",
    class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
    shuffle=True,
    seed=123,
    batch_size=batch_size,
    image_size=(32,32)
)

In [ ]:
valid_info = pd.DataFrame()

file_paths = valid.file_paths
valid_info['file_paths'] = file_paths


num_predictions = 15  # 預測次數
all_pred = []
all_pred_label = []

true_labels = [valid.class_names.index(label.split('/')[4]) for label in valid.file_paths]

for _ in range(num_predictions):
    prediction = model.predict(valid)
    prediction = tf.nn.softmax(prediction).numpy()
    all_pred.append(prediction)

    
all_pred_mean = np.mean(all_pred,axis=0)
all_pred_label = np.argmax(all_pred_mean, axis=1)

valid_info['pred_probs'] = np.max(all_pred_mean, axis=1)
valid_info['pred_labels'] = all_pred_label
valid_info['true_labels'] = true_labels
valid_info['label_quality'] = get_label_quality_scores(valid_info['true_labels'],all_pred_mean)


In [ ]:
label_issues = find_label_issues(valid_info['true_labels'], all_pred_mean, filter_by="both")

issue_valid = valid_info.iloc[label_issues]
issue_valid = pd.DataFrame(issue_valid)
issue_valid = issue_valid.sort_values('label_quality', ascending=True)
issue_valid = issue_valid.reset_index(drop=False)
issue_valid
print(f"Cleanlab found {len(issue_valid)} label issues.")

issue_valid

In [ ]:
classnames = valid.class_names

valid_info['ispreserve'] = 0

if not os.path.isdir('/kaggle/working/perserve/valid'):
    os.makedirs('/kaggle/working/perserve/valid')

for k in np.arange(0,len(issue_valid),15):
    plt.figure(figsize=(14,8))
    for idx, i in enumerate(range(k,min(k+15,len(issue_valid)))):
        file_path = issue_valid.iloc[i]['file_paths']
        pred_label = issue_valid.iloc[i]['pred_labels']
        true_label = issue_valid.iloc[i]['true_labels']
        index = issue_valid.iloc[i]['index']
        
        img = cv.imread(file_path)
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        plt.subplot(3,5,idx+1)
        plt.imshow(img)
        plt.title(f"true: {true_label+1}, pred : {pred_label+1}, index : {index}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    
    print('Is there any problem?')
    torf = input('True : 1 or False : 0')
    
    if torf == 1:

        print('Which indices you want to perserve ?\n'
             'Will be move in an perserve file.')
        perserve_arr = input('indices')
    
        for j in perserve_arr.split(','):
            j = int(j)
            perserve_label = valid_info.iloc[j]['true_labels']
            perserve_file = valid_info.iloc[j]['file_paths']
            
            if not path.isdir(path.join('/kaggle/working/perserve/valid',classnames[perserve_label])):
                os.makedirs(os.path.join('/kaggle/working/perserve/valid',classnames[perserve_label]))
                
            try:
                target_dir = os.path.join('/kaggle/working/perserve/valid',classnames[perserve_label])
                shutil.copy(perserve_file, target_dir)
                valid_info['ispreserve'][j] = 1
                
            except Exception as e:
                print(f"移動失敗: {e}")
            

In [ ]:
preserved_index = valid_info[valid_info['ispreserve']==1].index
if not path.isdir('kaggle/working/delete/valid'):
    os.makedirs('kaggle/working/delete/valid')
for idx, i in enumerate(preserved_index):
    file_path = valid_info.iloc[i]['file_paths']
    pred_label = valid_info.iloc[i]['pred_labels']
    true_label = valid_info.iloc[i]['true_labels']
    
    img = cv.imread(file_path)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.title(f"true: {true_label+1}, pred : {pred_label+1}, index : {index}")
    plt.axis("off")
    plt.show()

    print('Delete : 0 or Move : 1')
    dm = int(input())

    if dm==0:
        shutil.copy(file_path, 'kaggle/working/delete/valid')
        os.remove(file_path)
    else:
        print('Where you want move to')
        move = input('i,ii,iii, ... etc.')

        try:
            shutil.move(file_path, path.join('/kaggle/working/val',move))
            print('move sucess')

        except Exception as e:
            print('move fail')

# 找移動
**因Working不好儲存 直接使用資料集**

In [ ]:
train_route = '/kaggle/input/practical-innovative-analytics-data-science-2024/training_data/train'
valid_route = '/kaggle/input/practical-innovative-analytics-data-science-2024/training_data/val'

adjust_train = '/kaggle/input/the-true-true-final-adjust/train'
adjust_valid = '/kaggle/input/the-true-true-final-adjust/val'

classnames = ['i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x']

In [ ]:
df_original_train = pd.DataFrame(columns = classnames)
df_original_valid = pd.DataFrame(columns = classnames)

df_adjust_train = pd.DataFrame(columns = classnames)
df_adjust_valid = pd.DataFrame(columns = classnames)

In [ ]:
#把檔名路徑分出來

def makedata(data, route):
    for col in data.columns:
        temp = list()
        ro = glob.glob(os.path.join(route +'/'+ col, '*.png'))
        for r in ro:
            temp.append(os.path.split(r)[1])
        data[col] = pd.Series(temp)
    
    return data

df_original_train = makedata(df_original_train, train_route)

df_original_valid = makedata(df_original_valid, valid_route)

df_adjust_train = makedata(df_adjust_train, adjust_train)

df_adjust_valid = makedata(df_adjust_valid, adjust_valid)

In [ ]:
# 找不同

def diff(original, adjust):
    
    different = {}

    for col in classnames:
        deleted = set(original[col]) - set(adjust[col])
        bemoved = set(adjust[col]) - set(original[col])
    
    
        unique_diff = pd.DataFrame({
            "deleted": pd.Series(list(deleted)),
            "bemoved": pd.Series(list(bemoved))
        })

        different[col] = unique_diff
        
    return different

## Train

In [ ]:
diff_train = diff(df_original_train, df_adjust_train)

In [ ]:
for classname in classnames:
    print(f'The be moved image in {classname}')
    i=0
    for idx ,moved in enumerate(diff_train[classname]['bemoved']):

        if (moved.find('small') != -1) or (moved.find('cap') != -1) or (len(moved)>15):
            pass
        else:
            i+=1
            print(moved)
        
    print(f'We have {i} new image in {classname}\n')
        

In [ ]:
for classname in classnames:
    print(f'The deleted image in {classname}')
    i=0
    for row in diff_train[classname]['deleted'].dropna():
        i+=1
    print(f"We deleted {i} image in {classname}")

## Valid

In [ ]:
diff_valid = diff(df_original_valid, df_adjust_valid)

In [ ]:
for classname in classnames:
    print(f'The be moved image in {classname}')
    i=0
    for idx ,moved in enumerate(diff_valid[classname]['bemoved'].dropna()):

        if (moved.find('small') != -1) or (moved.find('cap') != -1) or (len(moved)>15):
            pass
        else:
            i+=1
            print(moved)
        
    print(f'We have {i} new image in {classname}\n')
        

In [ ]:
for classname in classnames:
    print(f'The deleted image in {classname}')
    i=0
    for row in diff_valid[classname]['deleted'].dropna():
        i+=1
    print(f"We deleted {i} image in {classname}")

# 增加資料量 by [The Chars74K dataset](http://info-ee.surrey.ac.uk/CVSSP/demos/chars74k/)

## 函數庫

In [ ]:
def gaussian_noise(image):
    noise = np.zeros((image.shape[0], image.shape[1]),dtype=np.uint8)
    cv.randn(noise, 128, 20)
    noise = (noise*random.uniform(0, 0.2)).astype(np.uint8)
    noise = cv.merge((noise,noise,noise))
    return noise

def uniform_noise(image):
    noise = np.zeros((image.shape[0], image.shape[1]),dtype=np.uint8)
    cv.randu(noise,0,255)
    noise = (noise*random.uniform(0, 0.2)).astype(np.uint8)
    noise = cv.merge((noise,noise,noise))
    return noise

def impulse_noise(image):
    noise = np.zeros((image.shape[0], image.shape[1]),dtype=np.uint8)
    ret,noise = cv.threshold(noise,250,255,cv.THRESH_BINARY)
    noise = (noise*random.uniform(0, 0.2)).astype(np.uint8)
    noise = cv.merge((noise,noise,noise))
    return noise

In [ ]:
def add_noise(img):
    img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
  
    # Getting the dimensions of the image
    row , col = img.shape
      
    # Randomly pick some pixels in the
    # image for coloring them white
    # Pick a random number between 300 and 10000
    number_of_pixels = random.randint(300, 100_000)
    for i in range(number_of_pixels):
        
        # Pick a random y coordinate
        y_coord=random.randint(0, row - 1)
          
        # Pick a random x coordinate
        x_coord=random.randint(0, col - 1)
          
        # Colour that pixel to white
#         img[y_coord][x_coord] = 255
        
        if (i%random.randint(10,20)==0 and 5<y_coord<895 and 5<x_coord<1195):
            a = random.randint(0,5)
            b = random.randint(0,5)
            yy=0
            xx=0
            while yy<a and xx<b:
                y_new = y_coord+yy
                x_new = x_coord+xx
                if 0<y_new<900 and 0<x_new<1200:
                    img[y_new][x_new] = 255
                else:
                    break
                yy+=random.randint(-2,2)
                xx+=random.randint(-2,2)
        
          
    # Randomly pick some pixels in
    # the image for coloring them black
    # Pick a random number between 300 and 10000
    number_of_pixels = random.randint(300 , 100_000)
    for i in range(number_of_pixels):
        
        # Pick a random y coordinate
        y_coord=random.randint(0, row - 1)
          
        # Pick a random x coordinate
        x_coord=random.randint(0, col - 1)
          
        # Colour that pixel to black
#         img[y_coord][x_coord] = 0
        
        if (i%random.randint(10,20)==0 and 5<y_coord<895 and 5<x_coord<1195):
            a = random.randint(0,5)
            b = random.randint(0,5)
            yy=0
            xx=0
            while yy<a and xx<b:
                y_new = y_coord+yy
                x_new = x_coord+xx
                if 0<y_new<900 and 0<x_new<1200:
                    img[y_new][x_new] = 0
                else:
                    break
                yy+=random.randint(-2,2)
                xx+=random.randint(-2,2)
        
    img = img.astype(np.uint8)
    img = cv.merge((img,img,img))
    return img

In [ ]:
def dilation(image, n=0):
    kernel= np.ones((3,3), np.uint8)
    if n==0:
        n = random.randint(2, 25)
    image_dilation = cv.dilate(image, kernel, iterations=n)
    return image_dilation,n

## Start

In [ ]:
os.makedirs('/kaggle/working/train', exist_ok=True)
os.makedirs('/kaggle/working/val', exist_ok=True)

### Capital

In [ ]:
# Capital 1s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    noisy_image1 = cv.add(dil_img,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/i', f'1_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making capital 2s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
#     img_with_border = cv.copyMakeBorder(img, 0, 0, 300, 0, cv.BORDER_CONSTANT, value=[255,255,255])
#     cv.imshow('img',dst)
    dst2 = cv.rectangle(dst, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dil_img, 0.5, dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/ii', f'2_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making capital 3s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = dil_img.shape[:2]
    M = np.float32([[1,0,200],[0,1,0]])
    M_M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    dst_dst = cv.warpAffine(dil_img,M_M,(cols,rows))
#     img_with_border = cv.copyMakeBorder(img, 0, 0, 300, 0, cv.BORDER_CONSTANT, value=[255,255,255])
#     cv.imshow('img',dst)
    dst2 = cv.rectangle(dst, (0,0), (200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst_dst, (1000,0), (1200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dil_img, 0.5, dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst2, 0.5, 0.0)
    mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
    dst_dst3[mask2>0]=(0,0,0)
    noisy_image1 = cv.add(dst_dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/iii', f'3_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making capital 4s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/iv', f'4_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Additional capital 4s
i=1
while i<56:
    if os.path.exists(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")==False:
        i+=1
        continue
        
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/iv', f'4_cap_copy_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    plt.imshow(noisy)
    plt.show()
    i+=1

In [ ]:
# Capital 5s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    noisy_image1 = cv.add(dil_img,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/v', f'5_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Additional capital 5s
i=1
while i<56:
    if os.path.exists(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")==False:
        i+=1
        continue
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")
    dil_img,n = dilation(img)
    noisy_image1 = cv.add(dil_img,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/v', f'5_cap_copy_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    plt.imshow(noisy)
    plt.show()
    i+=1

In [ ]:
# Making capital 6s
i=1
while i<56:
    if i == 49:
        img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}.png")
        rows,cols = img.shape[:2]
        M = np.float32([[1,0,-400],[0,1,0]])
        dst = cv.warpAffine(img,M,(cols,rows))
    
        img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
        M2 = np.float32([[1,0,0],[0,1,0]])
        dst2 = cv.warpAffine(img2,M2,(cols,rows))
    
        dst_dst = cv.rectangle(dst, (800,0), (1200,900), (255,255,255), -1)
        dst_dst2 = cv.rectangle(dst2, (-200,0), (200,900), (255,255,255), -1)
        dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
        mask=cv.inRange(dst3,(0,0,0),(130,130,130))
        dst3[mask>0]=(0,0,0)
        cv.imwrite(os.path.join('/kaggle/working/train/vi', f'6_cap_{str(i)}.png'), dst3)
        i+=1
        continue
    
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/vi', f'6_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Additional capital 6s
i=1
while i<56:
    if os.path.exists(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")==False:
        i+=1
        continue
    
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/vi', f'6_cap_copy_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    plt.imshow(noisy)
    plt.show()
    i+=1

In [ ]:
# Making capital 7s
i=1
while i<56:
    if i==49:
        img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}.png")
        rows,cols = img.shape[:2]
        M = np.float32([[1,0,-350],[0,1,0]])
        dst = cv.warpAffine(img,M,(cols,rows))
    
        img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
        M2 = np.float32([[1,0,0],[0,1,0]])
        dst2 = cv.warpAffine(img2,M2,(cols,rows))
        M22 = np.float32([[1,0,200],[0,1,0]])
        dst22 = cv.warpAffine(img2,M22,(cols,rows))
    
        dst_dst = cv.rectangle(dst, (800,0), (1200,900), (255,255,255), -1)
        dst_dst2 = cv.rectangle(dst2, (-200,0), (200,900), (255,255,255), -1)
        dst_dst22 = cv.rectangle(dst22, (-200,0), (400,900), (255,255,255), -1)
    
        dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
        mask=cv.inRange(dst3,(0,0,0),(130,130,130))
        dst3[mask>0]=(0,0,0)
    
        dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
        mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
        dst_dst3[mask2>0]=(0,0,0)
    
        cv.imwrite(os.path.join('/kaggle/working/train/vii', f'7_cap_{str(i)}.png'), dst_dst3)
        i+=1
        continue
        
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-150],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    M22 = np.float32([[1,0,400],[0,1,0]])
    dst22 = cv.warpAffine(dil_img2,M22,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst_dst22 = cv.rectangle(dst22, (0,0), (400,900), (255,255,255), -1)
    
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    
    dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
    mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
    dst_dst3[mask2>0]=(0,0,0)
    
    noisy_image1 = cv.add(dst_dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    
    cv.imwrite(os.path.join('/kaggle/working/train/vii', f'7_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Additional capital 7s
i=1
while i<56:
    if os.path.exists(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")==False:
        i+=1
        continue
    
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")    
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    M22 = np.float32([[1,0,400],[0,1,0]])
    dst22 = cv.warpAffine(dil_img2,M22,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst_dst22 = cv.rectangle(dst22, (0,0), (400,900), (255,255,255), -1)
    
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    
    dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
    mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
    dst_dst3[mask2>0]=(0,0,0)
    
    noisy_image1 = cv.add(dst_dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    
    cv.imwrite(os.path.join('/kaggle/working/train/vii', f'7_cap_copy_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    plt.imshow(noisy)
    plt.show()
    i+=1

In [ ]:
# Making capital 8s
i=1
while i<56:
    if i==49:
        img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}.png")
        rows,cols = img.shape[:2]
        M = np.float32([[1,0,-500],[0,1,0]])
        dst = cv.warpAffine(img,M,(cols,rows))
    
        img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
        M2 = np.float32([[1,0,-150],[0,1,0]])
        dst2 = cv.warpAffine(img2,M2,(cols,rows))
        M22 = np.float32([[1,0,50],[0,1,0]])
        dst22 = cv.warpAffine(img2,M22,(cols,rows))
        M23 = np.float32([[1,0,250],[0,1,0]])
        dst23 = cv.warpAffine(img2,M23,(cols,rows))
    
        dst_dst = cv.rectangle(dst, (700,0), (1200,900), (255,255,255), -1)
        dst_dst2 = cv.rectangle(dst2, (1000,0), (1200,900), (255,255,255), -1)
        dst_dst22 = cv.rectangle(dst22, (-250,0), (400,900), (255,255,255), -1)
        dst_dst23 = cv.rectangle(dst23, (-250,0), (600,900), (255,255,255), -1)
    
        dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
        mask=cv.inRange(dst3,(0,0,0),(130,130,130))
        dst3[mask>0]=(0,0,0)
    
        dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
        mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
        dst_dst3[mask2>0]=(0,0,0)
    
        dst_dst4 = cv.addWeighted(dst_dst3, 0.5, dst_dst23, 0.5, 0.0)
        mask3=cv.inRange(dst_dst4,(0,0,0),(130,130,130))
        dst_dst4[mask3>0]=(0,0,0)
    
        cv.imwrite(os.path.join('/kaggle/working/train/viii', f'8_cap_{str(i)}.png'), dst_dst4)
        i+=1
        continue
    
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-250],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,100],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    M22 = np.float32([[1,0,300],[0,1,0]])
    dst22 = cv.warpAffine(dil_img2,M22,(cols,rows))
    M23 = np.float32([[1,0,500],[0,1,0]])
    dst23 = cv.warpAffine(dil_img2,M23,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (950,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst_dst22 = cv.rectangle(dst22, (0,0), (400,900), (255,255,255), -1)
    dst_dst23 = cv.rectangle(dst23, (0,0), (600,900), (255,255,255), -1)
    
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    
    dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
    mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
    dst_dst3[mask2>0]=(0,0,0)
    
    dst_dst4 = cv.addWeighted(dst_dst3, 0.5, dst_dst23, 0.5, 0.0)
    mask3=cv.inRange(dst_dst4,(0,0,0),(130,130,130))
    dst_dst4[mask3>0]=(0,0,0)
    
    noisy_image1 = cv.add(dst_dst4,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    
    cv.imwrite(os.path.join('/kaggle/working/train/viii', f'8_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Additional capital 8s
i=1
while i<56:
    if os.path.exists(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")==False:
        i+=1
        continue
    
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img032-{str(i).zfill(3)}"
                    +' - Copy'+".png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-250],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,100],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    M22 = np.float32([[1,0,300],[0,1,0]])
    dst22 = cv.warpAffine(dil_img2,M22,(cols,rows))
    M23 = np.float32([[1,0,500],[0,1,0]])
    dst23 = cv.warpAffine(dil_img2,M23,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (950,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst_dst22 = cv.rectangle(dst22, (0,0), (400,900), (255,255,255), -1)
    dst_dst23 = cv.rectangle(dst23, (0,0), (600,900), (255,255,255), -1)
    
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    
    dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
    mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
    dst_dst3[mask2>0]=(0,0,0)
    
    dst_dst4 = cv.addWeighted(dst_dst3, 0.5, dst_dst23, 0.5, 0.0)
    mask3=cv.inRange(dst_dst4,(0,0,0),(130,130,130))
    dst_dst4[mask3>0]=(0,0,0)
    
    noisy_image1 = cv.add(dst_dst4,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    
    cv.imwrite(os.path.join('/kaggle/working/train/viii', f'8_cap_copy_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    plt.imshow(noisy)
    plt.show()
    i+=1

In [ ]:
# Making capital 9s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img034-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/ix', f'9_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Additional capital 9s
i=1
while i<56:
    if os.path.exists(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img034-{str(i).zfill(3)}"
                    +' - Copy'+".png")==False:
        i+=1
        continue
        
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img019-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img034-{str(i).zfill(3)}"
                    +' - Copy'+".png")
    
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    
    cv.imwrite(os.path.join('/kaggle/working/train/ix', f'9_cap_copy_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    plt.imshow(noisy)
    plt.show()
    i+=1

In [ ]:
# Capital 10s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img034-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    noisy_image1 = cv.add(dil_img,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/x', f'10_cap_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Additional capital 10s
i=1
while i<56:
    if os.path.exists(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img034-{str(i).zfill(3)}"
                    +' - Copy'+".png")==False:
        i+=1
        continue
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img034-{str(i).zfill(3)}"
                    +' - Copy'+".png")
    dil_img,n = dilation(img)
    noisy_image1 = cv.add(dil_img,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/x', f'10_cap_copy_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    plt.imshow(noisy)
    plt.show()
    i+=1

### Small

In [ ]:
# Small 1s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    noisy_image1 = cv.add(dil_img,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/i', f'1_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making small 2s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
#     img_with_border = cv.copyMakeBorder(img, 0, 0, 300, 0, cv.BORDER_CONSTANT, value=[255,255,255])
#     cv.imshow('img',dst)
    dst2 = cv.rectangle(dst, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dil_img, 0.5, dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/ii', f'2_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making small 3s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,200],[0,1,0]])
    M_M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    dst_dst = cv.warpAffine(dil_img,M_M,(cols,rows))
#     img_with_border = cv.copyMakeBorder(img, 0, 0, 300, 0, cv.BORDER_CONSTANT, value=[255,255,255])
#     cv.imshow('img',dst)
    dst2 = cv.rectangle(dst, (0,0), (200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst_dst, (1000,0), (1200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dil_img, 0.5, dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst2, 0.5, 0.0)
    mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
    dst_dst3[mask2>0]=(0,0,0)
    noisy_image1 = cv.add(dst_dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/iii', f'3_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making small 4s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img058-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/iv', f'4_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Small 5s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img058-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    noisy_image1 = cv.add(dil_img,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/v', f'5_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making small 6s
i=1
while i<56:
    if i == 49:
        img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img058-{str(i).zfill(3)}.png")
        rows,cols = img.shape[:2]
        M = np.float32([[1,0,-400],[0,1,0]])
        dst = cv.warpAffine(img,M,(cols,rows))
    
        img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
        M2 = np.float32([[1,0,0],[0,1,0]])
        dst2 = cv.warpAffine(img2,M2,(cols,rows))
    
        dst_dst = cv.rectangle(dst, (800,0), (1200,900), (255,255,255), -1)
        dst_dst2 = cv.rectangle(dst2, (-200,0), (200,900), (255,255,255), -1)
        dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
        mask=cv.inRange(dst3,(0,0,0),(130,130,130))
        dst3[mask>0]=(0,0,0)
        cv.imwrite(os.path.join('/kaggle/working/train/vi', f'6_small_{str(i)}.png'), dst3)
        i+=1
        continue
    
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img058-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/vi', f'6_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making small 7s
i=1
while i<56:
    if i==49:
        img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img058-{str(i).zfill(3)}.png")
        rows,cols = img.shape[:2]
        M = np.float32([[1,0,-350],[0,1,0]])
        dst = cv.warpAffine(img,M,(cols,rows))
    
        img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
        M2 = np.float32([[1,0,0],[0,1,0]])
        dst2 = cv.warpAffine(img2,M2,(cols,rows))
        M22 = np.float32([[1,0,200],[0,1,0]])
        dst22 = cv.warpAffine(img2,M22,(cols,rows))
    
        dst_dst = cv.rectangle(dst, (800,0), (1200,900), (255,255,255), -1)
        dst_dst2 = cv.rectangle(dst2, (-200,0), (200,900), (255,255,255), -1)
        dst_dst22 = cv.rectangle(dst22, (-200,0), (400,900), (255,255,255), -1)
    
        dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
        mask=cv.inRange(dst3,(0,0,0),(130,130,130))
        dst3[mask>0]=(0,0,0)
    
        dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
        mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
        dst_dst3[mask2>0]=(0,0,0)
    
        cv.imwrite(os.path.join('/kaggle/working/train/vii', f'7_small_{str(i)}.png'), dst_dst3)
        i+=1
        continue
        
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img058-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-150],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    M22 = np.float32([[1,0,400],[0,1,0]])
    dst22 = cv.warpAffine(dil_img2,M22,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst_dst22 = cv.rectangle(dst22, (0,0), (400,900), (255,255,255), -1)
    
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    
    dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
    mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
    dst_dst3[mask2>0]=(0,0,0)
    
    noisy_image1 = cv.add(dst_dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    
    cv.imwrite(os.path.join('/kaggle/working/train/vii', f'7_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making small 8s
i=1
while i<56:
    if i==49:
        img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img058-{str(i).zfill(3)}.png")
        rows,cols = img.shape[:2]
        M = np.float32([[1,0,-500],[0,1,0]])
        dst = cv.warpAffine(img,M,(cols,rows))
    
        img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
        M2 = np.float32([[1,0,-150],[0,1,0]])
        dst2 = cv.warpAffine(img2,M2,(cols,rows))
        M22 = np.float32([[1,0,50],[0,1,0]])
        dst22 = cv.warpAffine(img2,M22,(cols,rows))
        M23 = np.float32([[1,0,250],[0,1,0]])
        dst23 = cv.warpAffine(img2,M23,(cols,rows))
    
        dst_dst = cv.rectangle(dst, (700,0), (1200,900), (255,255,255), -1)
        dst_dst2 = cv.rectangle(dst2, (1000,0), (1200,900), (255,255,255), -1)
        dst_dst22 = cv.rectangle(dst22, (-250,0), (400,900), (255,255,255), -1)
        dst_dst23 = cv.rectangle(dst23, (-250,0), (600,900), (255,255,255), -1)
    
        dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
        mask=cv.inRange(dst3,(0,0,0),(130,130,130))
        dst3[mask>0]=(0,0,0)
    
        dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
        mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
        dst_dst3[mask2>0]=(0,0,0)
    
        dst_dst4 = cv.addWeighted(dst_dst3, 0.5, dst_dst23, 0.5, 0.0)
        mask3=cv.inRange(dst_dst4,(0,0,0),(130,130,130))
        dst_dst4[mask3>0]=(0,0,0)
    
        cv.imwrite(os.path.join('/kaggle/working/train/viii', f'8_small_{str(i)}.png'), dst_dst4)

        i+=1
        continue
    
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img058-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-250],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,50],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    M22 = np.float32([[1,0,250],[0,1,0]])
    dst22 = cv.warpAffine(dil_img2,M22,(cols,rows))
    M23 = np.float32([[1,0,450],[0,1,0]])
    dst23 = cv.warpAffine(dil_img2,M23,(cols,rows))
    
    dst_dst = cv.rectangle(dst, (950,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst_dst22 = cv.rectangle(dst22, (0,0), (400,900), (255,255,255), -1)
    dst_dst23 = cv.rectangle(dst23, (0,0), (600,900), (255,255,255), -1)
    
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    
    dst_dst3 = cv.addWeighted(dst3, 0.5, dst_dst22, 0.5, 0.0)
    mask2=cv.inRange(dst_dst3,(0,0,0),(130,130,130))
    dst_dst3[mask2>0]=(0,0,0)
    
    dst_dst4 = cv.addWeighted(dst_dst3, 0.5, dst_dst23, 0.5, 0.0)
    mask3=cv.inRange(dst_dst4,(0,0,0),(130,130,130))
    dst_dst4[mask3>0]=(0,0,0)
    
    noisy_image1 = cv.add(dst_dst4,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    
    cv.imwrite(os.path.join('/kaggle/working/train/viii', f'8_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Making small 9s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img045-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    rows,cols = img.shape[:2]
    M = np.float32([[1,0,-200],[0,1,0]])
    dst = cv.warpAffine(dil_img,M,(cols,rows))
    img2 = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img060-{str(i).zfill(3)}.png")
    dil_img2,n = dilation(img2,n)
    M2 = np.float32([[1,0,200],[0,1,0]])
    dst2 = cv.warpAffine(dil_img2,M2,(cols,rows))
    dst_dst = cv.rectangle(dst, (1000,0), (1200,900), (255,255,255), -1)
    dst_dst2 = cv.rectangle(dst2, (0,0), (200,900), (255,255,255), -1)
    dst3 = cv.addWeighted(dst_dst, 0.5, dst_dst2, 0.5, 0.0)
    mask=cv.inRange(dst3,(0,0,0),(130,130,130))
    dst3[mask>0]=(0,0,0)
    noisy_image1 = cv.add(dst3,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/ix', f'9_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

In [ ]:
# Small 10s
i=1
while i<56:
    img = cv.imread(f"../input/english-handwritten-characters-modified/English Handwritten Characters Modified/img060-{str(i).zfill(3)}.png")
    dil_img,n = dilation(img)
    noisy_image1 = cv.add(dil_img,gaussian_noise(img))
    noisy_image2 = cv.add(noisy_image1,uniform_noise(img))
    noisy_image3 = cv.add(noisy_image2,impulse_noise(img))
    noisy = add_noise(noisy_image3)
    cv.imwrite(os.path.join('/kaggle/working/train/x', f'10_small_{str(i)}.png'), noisy,  [cv.IMWRITE_PNG_COMPRESSION, 9])
    if i<6:
        plt.imshow(noisy)
        plt.show()
    i+=1

# 分類增強

## 略過前置作業
**因Working不好儲存**

In [ ]:
restart()

In [ ]:
shutil.rmtree('/kaggle/working/clean')
shutil.rmtree('/kaggle/working/2_augmented_v05')

In [ ]:
# 移動到working
ori_pos = '/kaggle/input/the-true-true-final-adjust'
wanttogo = '/kaggle/working'
shutil.copytree(os.path.join(ori_pos,'train'), 
                os.path.join(wanttogo, 'train'), ignore=shutil.ignore_patterns('*.ini'), dirs_exist_ok=True)

shutil.copytree(os.path.join(ori_pos,'val'), 
                os.path.join(wanttogo, 'val'), ignore=shutil.ignore_patterns('*.ini'), dirs_exist_ok=True)

CLEAN_DATA_FOLDER = '/kaggle/working/clean'
TRAIN_DATA = f'{CLEAN_DATA_FOLDER}/train'
VALID_DATA = f'{CLEAN_DATA_FOLDER}/val'

shutil.copytree('/kaggle/working/', CLEAN_DATA_FOLDER, dirs_exist_ok=True)


In [ ]:
# # Rename files in train and val sets (in new clean folder) for easier tracking
# data_types = [TRAIN_DATA, VALID_DATA]

# for data_type in data_types:
#     for folder in os.listdir(data_type):
#           for index, file in enumerate(os.listdir(data_type + '/' + folder)):
#                 data_type_name = data_type.split('/')[-1]
#                 os.rename(os.path.join(data_type, folder, file), os.path.join(data_type, folder, ''.join([str(data_type_name), '_', str(folder), '_', str(index),'.png'])))

In [ ]:
# 顯示轉換後的圖像比較
def show_single_transform(transform, show_histogram=False):
    images, filenames = [], []
    
    for folder in os.listdir(TRAIN_DATA):
          for image in os.listdir(TRAIN_DATA + '/' + folder):
            images.append(os.path.join(TRAIN_DATA, folder, image))
            filenames.append(f'{image}')
            
    random_index = random.choice(range(len(images)))
    random_filename = filenames[random_index]
    random_img = images[random_index]
    
    img_original = Image.open(random_img)
    img_original = ImageOps.grayscale(img_original) # applying greyscale method

    # Execute transformation
    try:
        img_transformed = transform(img_original)
    except:
        img_transformed = transform(images=np.asarray(img_original))
    print(f'Transformation successful')
        
    # Display images side by side
    fig = plt.figure(figsize=(14, 8))

    # show original image
    fig.add_subplot(221)
    plt.title('Original Image')
    plt.axis('off')
    plt.imshow(img_original, cmap=plt.get_cmap('gray'))
    fig.add_subplot(222)
    plt.title('Transformed Image')
    plt.axis('off')
    plt.imshow(img_transformed, cmap=plt.get_cmap('gray'))
    
    if show_histogram==True:
        hist_original = cv2.calcHist([np.asarray(img_original)],[0],None,[256],[0,256])
        hist_transformed = cv2.calcHist([np.asarray(img_transformed)],[0],None,[256],[0,256])
        
        fig.add_subplot(223)
        plt.title('Histogram of Original')
        plt.plot(hist_original)
        fig.add_subplot(224)
        plt.title('Histogram of Transformed')
        plt.plot(hist_transformed)
    
    plt.show();

In [ ]:
def augment_images_shuffle(label, transform, aug_version, 
                           total_size=1200, train_size=700,
                           extra_transform=None):
    # 設定相關的資料夾路徑
    input_folder_train = f'{CLEAN_DATA_FOLDER}/train/{label}'
    input_folder_valid = f'{CLEAN_DATA_FOLDER}/val/{label}'
    
    # 建立專屬於該標籤的暫存資料夾
    temp_folder = f'/kaggle/working/2_augmented_{aug_version}/temp_{label}'
    Path(temp_folder).mkdir(parents=True, exist_ok=True)
   
    # 將所有乾淨的圖片複製到暫存資料夾
    input_folders = [input_folder_train, input_folder_valid]
    for input_folder in input_folders:
        for image in os.listdir(input_folder):
            shutil.copy(f'{input_folder}/{image}', temp_folder)

    # 建立輸出資料夾路徑
    output_folder_train = f'/kaggle/working/2_augmented_{aug_version}/train/{label}'
    output_folder_val = f'/kaggle/working/2_augmented_{aug_version}/val/{label}'
    Path(output_folder_train).mkdir(parents=True, exist_ok=True)
    Path(output_folder_val).mkdir(parents=True, exist_ok=True)
        
    # 獲取有效的圖片檔案清單
    input_files = glob.glob(os.path.join(temp_folder, "*.png"))
    valid_files = []
    for file in input_files:
        try:
            with Image.open(file) as img:
                img.verify()  # 驗證檔案是否為有效圖片
                valid_files.append(file)
        except UnidentifiedImageError:
            print(f"跳過無效圖片: {file}")
        except Exception as e:
            print(f"驗證圖片 {file} 時發生錯誤: {e}")

    print(f'建立暫存資料夾: {temp_folder}。有效圖片數量: {len(valid_files)}')

    # 計算需要額外生成的圖片數量
    temp_folder_count = len(valid_files)
    balance_count = total_size - temp_folder_count

    n = 0
    for i in range(balance_count):
        try:
            # 隨機選取一張圖片進行增強
            random_file = random.choice(valid_files)
            img_random = Image.open(random_file).convert('L')  # 轉為灰階
            img_array = np.asarray(img_random)  # 轉為 NumPy 陣列

            # 執行增強轉換
            transformed_img_random = transform(images=img_array)

            # 如果有提供額外的增強方法，則執行
            if extra_transform is not None:
                transformed_img_random = extra_transform(transformed_img_random)

            # 轉回 PIL 圖片並儲存
            transformed_img_random = Image.fromarray(transformed_img_random)
            n += 1
            transformed_img_random.save(f'{temp_folder}/{label}_random_{n}.png', 'PNG')

        except Exception as e:
            print(f"處理圖片 {random_file} 時發生錯誤: {e}")
            continue

    # 將圖片分割為訓練集與驗證集
    full_img_list = os.listdir(temp_folder)

    if len(full_img_list) < train_size:
        print(f"訓練集圖片不足。調整 train_size 至 {len(full_img_list) * 0.8:.0f}")
        train_size = int(len(full_img_list) * 0.8)  # 使用 80% 作為訓練集
        val_size = len(full_img_list) - train_size
    else:
        val_size = len(full_img_list) - train_size

    train_list = random.sample(full_img_list, train_size)
    val_list = [x for x in full_img_list if x not in train_list]

    for file in train_list:
        shutil.copy(os.path.join(temp_folder, file), output_folder_train)
    for file in val_list:
        shutil.copy(os.path.join(temp_folder, file), output_folder_val)
    
    # 清除暫存資料夾
    shutil.rmtree(temp_folder, ignore_errors=True)
    print(f"標籤 '{label}' 的圖片增強完成。訓練集: {len(train_list)}，驗證集: {len(val_list)}")


In [ ]:
# Refined Transform v5a1
transform_v5a = iaa.Sequential([
    iaa.Fliplr(0.5),  # Horizontal flip
    iaa.Flipud(0.2),  # Vertical flip (less frequent)
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.05)),  # Small crop
    iaa.GaussianBlur(sigma=(0, 3)),  # Moderate blur
    iaa.LinearContrast((1.2, 1.5)),  # Adjust contrast
    iaa.Multiply((0.85, 1.15), per_channel=0.3),  # Adjust brightness
    iaa.Affine(
        scale={"x": (0.9, 1.1), "y": (0.9, 1.1)},
        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
        rotate=(-15, 15),
        shear=(-3, 3),
        cval=255
    )
], random_order=False)

# Refined Transform v5b2
transform_v5b = iaa.Sequential([
    iaa.Fliplr(0.5),
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.05)),
    iaa.AdditiveGaussianNoise(scale=(0, 0.5 * 255)),  # Light noise
    iaa.LinearContrast((1.2, 1.4)),
    iaa.Multiply((0.9, 1.1), per_channel=0.2),
    iaa.Affine(
        scale={"x": (0.9, 1.1), "y": (0.9, 1.1)},
        translate_percent={"x": (-0.04, 0.04), "y": (-0.04, 0.04)},
        rotate=(-10, 10),
        shear=(-2, 2),
        cval=255
    )
], random_order=False)

# Refined Transform v5c3
transform_v5c = iaa.Sequential([
    iaa.Fliplr(0.5),
    iaa.Flipud(0.2),
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.03)),
    iaa.GaussianBlur(sigma=(0, 2)),
    iaa.LinearContrast((1.1, 1.4)),
    iaa.AdditiveGaussianNoise(loc=0, scale=(0.0, 0.5 * 255), per_channel=0.2),
    iaa.Multiply((0.9, 1.2), per_channel=0.2),
    iaa.Affine(
        scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
        translate_percent={"x": (-0.04, 0.04), "y": (-0.04, 0.04)},
        rotate=(-10, 10),
        shear=(-3, 3),
        cval=255
    )
], random_order=False)

# Refined Transform v5d4
transform_v5d = iaa.Sequential([
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.03)),
    iaa.GaussianBlur(sigma=(0, 1)),
    iaa.LinearContrast((1.2, 1.5)),
    iaa.Multiply((0.85, 1.15), per_channel=0.3),
    iaa.Affine(
        scale={"x": (0.9, 1.1), "y": (0.9, 1.1)},
        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
        rotate=(-15, 15),
        shear=(-4, 4),
        cval=255
    )
], random_order=False)

# Refined Transform v5e5
transform_v5e = iaa.Sequential([
    iaa.Fliplr(0.25),
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.05)),
    iaa.GaussianBlur(sigma=(0, 0.4)),
    iaa.LinearContrast((0.7, 1.4)),
    iaa.Multiply((0.55, 1.2), per_channel=0.3),
    iaa.Affine(
        scale={"x": (0.9, 1.15), "y": (0.9, 1.15)},
        translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
        rotate=(-10, 10),
        shear=(-2, 2),
        cval=255
    )
], random_order=False)

# Refined Transform v5f6
transform_v5f = iaa.Sequential([
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.03)),
    iaa.GaussianBlur(sigma=(0, 0.5)),
    iaa.LinearContrast((1.1, 1.4)),
    iaa.Multiply((0.9, 1.1), per_channel=0.2),
    iaa.Affine(
        scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
        translate_percent={"x": (-0.08, 0.08), "y": (-0.08, 0.08)},
        rotate=(-20, 20),
        shear=(-5, 5),
        cval=255
    )
], random_order=False)

# Refined Transform v5g7
transform_v5g = iaa.Sequential([
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.03)),
    iaa.GaussianBlur(sigma=(0, 1)),
    iaa.LinearContrast((1.1, 1.4)),
    iaa.Multiply((0.9, 1.1), per_channel=0.2),
    iaa.Affine(
        scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
        translate_percent={"x": (-0.08, 0.08), "y": (-0.08, 0.08)},
        rotate=(-20, 20),
        shear=(-5, 5),
        cval=255
    )
], random_order=False)

# Refined Transform v5h8
transform_v5h = iaa.Sequential([
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.05)),
    iaa.GaussianBlur(sigma=(0, 1.5)),
    iaa.LinearContrast((1.2, 1.6)),
    iaa.Multiply((0.85, 1.15), per_channel=0.3),
    iaa.Affine(
        scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
        translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
        rotate=(-15, 15),
        shear=(-4, 4),
        cval=255
    )
], random_order=False)

# Refined Transform v5i9
transform_v5i = iaa.Sequential([
    iaa.Flipud(0.5),
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.05)),
    iaa.GaussianBlur(sigma=(0, 0.7)),
    iaa.LinearContrast((1.2, 1.6)),
    iaa.AdditiveGaussianNoise(loc=0, scale=(0.0, 0.05 * 255), per_channel=0.3),
    iaa.Multiply((0.85, 1.15), per_channel=0.3),
    iaa.Affine(
        scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
        translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
        rotate=(-15, 15),
        shear=(-4, 4),
        cval=255
    )
], random_order=False)

# Refined Transform v5j10
transform_v5j = iaa.Sequential([
    iaa.Flipud(0.5),
    iaa.Fliplr(0.5),
    iaa.Resize({"height": 400, "width": 400}),
    iaa.Crop(percent=(0, 0.05)),
    iaa.AdditiveGaussianNoise(loc=0, scale=(0.0, 0.85 * 255), per_channel=0.5),
    iaa.LinearContrast((1.2, 1.6)),
    iaa.Multiply((0.85, 1.15), per_channel=0.3),
    iaa.Affine(
        scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
        translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
        rotate=(-15, 15),
        shear=(-4, 4),
        cval=255
    )
], random_order=False)


In [ ]:
# 把每個類別分開增強
target_labels = ['i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x']
transforms = [transform_v5a, transform_v5b, transform_v5c, transform_v5d, 
              transform_v5e, transform_v5f, transform_v5g, transform_v5h, 
              transform_v5i, transform_v5j]


for label, transform in zip(target_labels, transforms):
    augment_images_shuffle(
        label=label,
        transform=transform,
        aug_version='v05',
        total_size=1200,
        train_size=1100
    )

In [ ]:
# 讀取檔案路徑
best_train_path = '/kaggle/working/2_augmented_v05/train'
best_valid_path = '/kaggle/working/2_augmented_v05/val'

user_data = '/kaggle/working/2_augmented_v05'

## 出問題專用

In [ ]:
# 檢查錯誤
def check_image_files(directory):
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    img.verify()  
            except UnidentifiedImageError:
                print(f"Invalid image file: {file_path}")
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")

# 檢查訓練和驗證數據集
check_image_files(best_train_path)
check_image_files(best_valid_path)

In [ ]:
def delete_invalid_images(directory):
    """檢查並刪除目錄中的無效圖像文件"""
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                # 嘗試打開圖像以驗證其有效性
                with Image.open(file_path) as img:
                    img.verify()  # 檢查圖像是否損壞
            except (UnidentifiedImageError, OSError):
                # 如果圖像無效，刪除文件
                print(f"Deleting invalid image: {file_path}")
                os.remove(file_path)
            except Exception as e:
                print(f"Unexpected error with file {file_path}: {e}")

# 指定需要清理的目錄
delete_invalid_images(best_train_path)
delete_invalid_images(best_valid_path)


In [ ]:
def count_images(directory, extensions=('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
    """計算資料夾中符合擴展名的圖片數量"""
    image_count = 0
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(extensions):  # 檢查檔案擴展名
                image_count += 1
    return image_count

# 指定目錄
train_image_count = count_images(user_data + '/train')
valid_image_count = count_images(user_data + '/val')

print(f"Train folder contains {train_image_count} images.")
print(f"Validation folder contains {valid_image_count} images.")

In [ ]:
def clean_invalid_files_forced(folder_path):
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'rb') as f:
                    f.read()  # 嘗試讀取文件
                with Image.open(file_path) as img:
                    img.verify()
            except Exception as e:
                print(f"Force removing invalid file: {file_path}, Error: {e}")
                os.remove(file_path)

clean_invalid_files_forced(user_data + '/train')
clean_invalid_files_forced(user_data + '/val')


In [ ]:
import hashlib

# 計算文件哈希值
def get_file_hash(file_path):
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

# 檢查並刪除重複文件
def remove_duplicates(folder_path):
    seen_hashes = set()
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            file_hash = get_file_hash(file_path)
            if file_hash in seen_hashes:
                print(f"Removing duplicate file: {file_path}")
                os.remove(file_path)
            else:
                seen_hashes.add(file_hash)

# 檢查訓練和驗證資料夾
remove_duplicates(user_data + '/train')
remove_duplicates(user_data + '/val')


# 造完資料集 fit model

In [ ]:
### DO NOT MODIFY BELOW THIS LINE, THIS IS THE FIXED MODEL ###
tf.keras.backend.clear_session()
batch_size = 8
tf.random.set_seed(2024)

train = tf.keras.preprocessing.image_dataset_from_directory(
        user_data + '/train',
        labels="inferred",
        label_mode="categorical",
        class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
        shuffle=True,
        seed=123,
        batch_size=batch_size,
        image_size=(32, 32),
    )

valid = tf.keras.preprocessing.image_dataset_from_directory(
        user_data + '/val',
        labels="inferred",
        label_mode="categorical",
        class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
        shuffle=True,
        seed=123,
        batch_size=batch_size,
        image_size=(32, 32),
)

total_length = ((train.cardinality() + valid.cardinality()) * batch_size).numpy()

if total_length > 12_000:
    print(f"Dataset size larger than 12,000. Got {total_length} examples")
    sys.exit()

test = tf.keras.preprocessing.image_dataset_from_directory(
        test_data,
        labels="inferred",
        label_mode="categorical",
        class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
        shuffle=False,
        seed=123,
        batch_size=batch_size,
        image_size=(32, 32),
)

# Initialize the base model using KerasCV's ResNet50
backbone = keras_cv.models.ResNet50Backbone.from_preset(
    input_shape=(32, 32, 3),
    preset = "resnet50_imagenet",
    load_weights=False,
)

# Create a new model that outputs the desired intermediate layer
base_model = tf.keras.Model(
    inputs=backbone.inputs,
    outputs=backbone.get_layer("v2_stack_0_block3_out").output
)

# Define the input tensor
inputs = tf.keras.Input(shape=(32, 32, 3))

# Pass the preprocessed input through the base model
x = base_model(inputs)

# Add global average pooling
x = tf.keras.layers.GlobalAveragePooling2D()(x)

# Add a dense layer for classification (assuming 10 classes)
x = tf.keras.layers.Dense(10)(x)

# Define the final model
model = tf.keras.Model(inputs, x)

# Compile the model with appropriate optimizer, loss, and metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

# Display the model's architecture
model.summary()
    
loss_0, acc_0 = model.evaluate(valid)
print(f"loss {loss_0}, acc {acc_0}")

checkpoint = tf.keras.callbacks.ModelCheckpoint(
        "best_model.weights.h5",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=3, verbose=1, min_lr=1e-7)

history = model.fit(
        train,
        validation_data=valid,
        epochs=75,
        callbacks=[checkpoint, lr_scheduler],
)

model.load_weights("best_model.weights.h5")

loss, acc = model.evaluate(valid)
print(f"final loss {loss}, final acc {acc}")

test_loss, test_acc = model.evaluate(test)
print(f"test loss {test_loss}, test acc {test_acc}")

### DO NOT MODIFY ABOVE THIS LINE, THIS IS THE FIXED MODEL ###

In [ ]:
model.evaluate(test)

In [ ]:
test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    directory+"testing_data",
    shuffle = False,
    image_size=(32, 32),
    batch_size=1)

prob = model.predict(test_dataset)
predictions = []
for i in range(0, prob.shape[0]):
    predictions.append(np.argmax(prob[i,:])+1)

In [ ]:
import pandas as pd

paths = test_dataset.file_paths

Ids = []
for x in paths:
    Ids.append(x.split("/")[-1])
    
df = pd.DataFrame()
df["Id"] = Ids
df["Predicted"] = predictions
df.to_csv("submission.csv", index=False)

# 下載檔案

In [ ]:
import os
import subprocess
from IPython.display import FileLink, display

def download_file(path, download_file_name):
    os.chdir('/kaggle/working/')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return
    display(FileLink(f'{download_file_name}.zip'))

download_file('/kaggle/working/train', 'train')

# 以下為調整參數時使用

In [ ]:
from sklearn.metrics import classification_report
predictedlabel = pd.read_csv('/kaggle/working/submission.csv')
print(classification_report(thetruelabel['label'], predictedlabel['Predicted']))

In [ ]:
thetruelabel = pd.read_csv('/kaggle/input/truelabel/mapping_test (1).csv')

In [ ]:
def editweight():
    target_labels = ['i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x']
    transforms = [transform_v5a, transform_v5b, transform_v5c, transform_v5d, 
                  transform_v5e, transform_v5f, transform_v5g, transform_v5h, 
                  transform_v5i, transform_v5j]
    
    for label, transform in zip(target_labels, transforms):
        if label == 'viii' :
            augment_images_shuffle(
                label=label,
                transform=transform,
                aug_version='v05',
                total_size=1200,
                train_size=800
            )

        elif label == 'v':
            augment_images_shuffle(
                label=label,
                transform=transform,
                aug_version='v05',
                total_size=1200,
                train_size=800
            )

        elif label == 'vi':
            augment_images_shuffle(
                label=label,
                transform=transform,
                aug_version='v05',
                total_size=1200,
                train_size=800
            )
        
        else:
            augment_images_shuffle(
                label=label,
                transform=transform,
                aug_version='v05',
                total_size=1200,
                train_size=800
            )

In [ ]:
# from imgaug import augmenters as iaa

# # Refined Transform v5a i
# transform_v5a = iaa.Sequential([
#     iaa.Fliplr(0.5),  # Horizontal flip
#     iaa.Flipud(0.2),  # Vertical flip (less frequent)
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.05)),  # Small crop
#     iaa.Sometimes(0.5, iaa.GaussianBlur(sigma=(0, 0.8))),  # Moderate blur
#     iaa.LinearContrast((1.2, 1.5)),  # Adjust contrast
#     iaa.Multiply((0.85, 1.15), per_channel=0.3),  # Adjust brightness
#     iaa.Affine(
#         scale={"x": (0.9, 1.1), "y": (0.9, 1.1)},
#         translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
#         rotate=(-15, 15),
#         shear=(-3, 3),
#         cval=255
#     )
# ], random_order=False)

# # Refined Transform v5b　ii
# transform_v5b = iaa.Sequential([
#     iaa.Fliplr(0.5),
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.05)),
#     iaa.Sometimes(0.4, iaa.AdditiveGaussianNoise(scale=(0, 0.02 * 255))),  # Light noise
#     iaa.LinearContrast((1.2, 1.4)),
#     iaa.Multiply((0.9, 1.1), per_channel=0.2),
#     iaa.Affine(
#         scale={"x": (0.9, 1.1), "y": (0.9, 1.1)},
#         translate_percent={"x": (-0.04, 0.04), "y": (-0.04, 0.04)},
#         rotate=(-10, 10),
#         shear=(-2, 2),
#         cval=255
#     )
# ], random_order=False)

# # Refined Transform v5c iii
# transform_v5c = iaa.Sequential([
#     iaa.Fliplr(0.5),
#     iaa.Flipud(0.2),
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.03)),
#     iaa.Sometimes(0.6, iaa.GaussianBlur(sigma=(0, 0.5))),
#     iaa.LinearContrast((1.1, 1.4)),
#     iaa.Multiply((0.9, 1.2), per_channel=0.2),
#     iaa.Affine(
#         scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
#         translate_percent={"x": (-0.04, 0.04), "y": (-0.04, 0.04)},
#         rotate=(-10, 10),
#         shear=(-3, 3),
#         cval=255
#     )
# ], random_order=False)

# # Refined Transform v5d iv
# transform_v5d = iaa.Sequential([
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.03)),
#     iaa.Sometimes(0.4, iaa.GaussianBlur(sigma=(0, 0.6))),
#     iaa.LinearContrast((1.2, 1.5)),
#     iaa.Multiply((0.85, 1.15), per_channel=0.3),
#     iaa.Affine(
#         scale={"x": (0.9, 1.1), "y": (0.9, 1.1)},
#         translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
#         rotate=(-15, 15),
#         shear=(-4, 4),
#         cval=255
#     )
# ], random_order=False)

# # Refined Transform v5e v
# transform_v5e = iaa.Sequential([
#                     iaa.Fliplr(0.5), # Horizontal flip
#                     iaa.Resize({"height": 400, "width": 400}),
#                     # iaa.Crop(percent=(0, 0.05)),
#                     iaa.Sometimes(
#                         0.25,
#                         iaa.GaussianBlur(sigma=(0, 1))
#                     ),
#                     iaa.LinearContrast((1.35, 1.75)),
#                     # iaa.AdditiveGaussianNoise(loc=0, 
#                     #                           scale=(0.0, 0.05*255), 
#                     #                           per_channel=0.5),
#                     # iaa.Multiply((0.8, 1.2), per_channel=0.2),
#                     iaa.Affine(
#                         scale={"x": (0.8, 1.2), "y": (0.8, 1.2)},
#                         translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
#                         # rotate=(-20, 20),#20,20
#                         shear=(-3, 3),
#                         # cval=255 # Fill with white pixels
#                     )
#                 ], random_order=False)

# # Refined Transform v5f  vi
# transform_v5f = iaa.Sequential([
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.03)),
#     iaa.Sometimes(0.4, iaa.GaussianBlur(sigma=(0, 1))),
#     iaa.LinearContrast((1.1, 1.4)),
#     iaa.Multiply((0.9, 1.1), per_channel=0.2),
#     iaa.Affine(
#         scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
#         translate_percent={"x": (-0.08, 0.08), "y": (-0.08, 0.08)},
#         # rotate=(-20, 20),
#         shear=(-5, 5),
#         cval=255
#     )
# ], random_order=False)

# # Refined Transform v5g vii
# transform_v5g = iaa.Sequential([ 
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.03)),
#     iaa.Sometimes(0.4, iaa.GaussianBlur(sigma=(0, 1))),
#     iaa.LinearContrast((1.1, 1.4)),
#     iaa.Multiply((0.9, 1.1), per_channel=0.2),
#     iaa.Affine(
#         scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
#         translate_percent={"x": (-0.08, 0.08), "y": (-0.08, 0.08)},
#         rotate=(-20, 20),
#         shear=(-5, 5),
#         cval=255
#     )
# ], random_order=False)

# # Refined Transform v5h viii
# transform_v5h = iaa.Sequential([
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.05)),
#     iaa.Sometimes(0.5, iaa.GaussianBlur(sigma=(0, 0.7))),
#     iaa.LinearContrast((1.2, 1.6)),
#     iaa.Multiply((0.85, 1.15), per_channel=0.3),
#     iaa.Affine(
#         scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
#         translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
#         rotate=(-15, 15),
#         shear=(-4, 4),
#         cval=255
#     )
# ], random_order=False)

# # Refined Transform v5i ix
# transform_v5i = iaa.Sequential([
#     iaa.Flipud(0.5),
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.05)),
#     iaa.Sometimes(0.5, iaa.GaussianBlur(sigma=(0, 0.7))),
#     iaa.LinearContrast((1.2, 1.6)),
#     iaa.Multiply((0.85, 1.15), per_channel=0.3),
#     iaa.Affine(
#         scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
#         translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
#         rotate=(-15, 15),
#         shear=(-4, 4),
#         cval=255
#     )
# ], random_order=False)

# # Refined Transform v5j x
# transform_v5j = iaa.Sequential([
#     iaa.Flipud(0.5),
#     iaa.Fliplr(0.5),
#     iaa.Resize({"height": 400, "width": 400}),
#     iaa.Crop(percent=(0, 0.05)),
#     iaa.Sometimes(0.5, iaa.GaussianBlur(sigma=(0, 0.7))),
#     iaa.LinearContrast((1.2, 1.6)),
#     iaa.Multiply((0.85, 1.15), per_channel=0.3),
#     iaa.Affine(
#         scale={"x": (0.85, 1.15), "y": (0.85, 1.15)},
#         translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
#         rotate=(-15, 15),
#         shear=(-4, 4),
#         cval=255
#     )
# ], random_order=False)

In [ ]:
# 6 Transform v5f
transform_v5f = iaa.Sequential([
                    iaa.Resize({"height": 400, "width": 400}),
                    iaa.Crop(percent=(0, 0.05)),
                    iaa.Sometimes(
                        0.5,
                        iaa.GaussianBlur(sigma=(0, 0.5))
                    ),
                    iaa.LinearContrast((1.35, 1.75)), 
                    iaa.AdditiveGaussianNoise(loc=0, scale=(0.0, 0.05*255), per_channel=0.25),
                    iaa.Multiply((0.8, 1.2), per_channel=0.2),
                    iaa.Affine(
                        scale={"x": (0.8, 1.2), "y": (0.8, 1.2)},
                        translate_percent={"x": (-0.06, 0.06), "y": (-0.06, 0.06)},
                        rotate=(-20, 20),
                        shear=(-3, 3),
                        cval=255
                    )
                ], random_order=False) # Cancel random order

In [ ]:
shutil.rmtree('/kaggle/working/clean')
shutil.rmtree('/kaggle/working/2_augmented_v05')


# 移動到working
ori_pos = '/kaggle/input/the-true-final-adjust'
wanttogo = '/kaggle/working'
shutil.copytree(os.path.join(ori_pos,'train'), 
                os.path.join(wanttogo, 'train'), ignore=shutil.ignore_patterns('*.ini'), dirs_exist_ok=True)

shutil.copytree(os.path.join(ori_pos,'val'), 
                os.path.join(wanttogo, 'val'), ignore=shutil.ignore_patterns('*.ini'), dirs_exist_ok=True)

# 定義清理後的資料夾路徑
CLEAN_DATA_FOLDER = '/kaggle/working/clean'
TRAIN_DATA = f'{CLEAN_DATA_FOLDER}/train'
VALID_DATA = f'{CLEAN_DATA_FOLDER}/val'

# 創建新資料夾並拷貝數據
shutil.copytree('/kaggle/working/', CLEAN_DATA_FOLDER, dirs_exist_ok=True)


editweight()
user_data = '/kaggle/working/2_augmented_v05'

In [ ]:
### DO NOT MODIFY BELOW THIS LINE, THIS IS THE FIXED MODEL ###
tf.keras.backend.clear_session()
batch_size = 8
tf.random.set_seed(2024)

train = tf.keras.preprocessing.image_dataset_from_directory(
        user_data + '/train',
        labels="inferred",
        label_mode="categorical",
        class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
        shuffle=True,
        seed=123,
        batch_size=batch_size,
        image_size=(32, 32),
    )

valid = tf.keras.preprocessing.image_dataset_from_directory(
        user_data + '/val',
        labels="inferred",
        label_mode="categorical",
        class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
        shuffle=True,
        seed=123,
        batch_size=batch_size,
        image_size=(32, 32),
)

total_length = ((train.cardinality() + valid.cardinality()) * batch_size).numpy()

if total_length > 12_000:
    print(f"Dataset size larger than 12,000. Got {total_length} examples")
    sys.exit()

test = tf.keras.preprocessing.image_dataset_from_directory(
        test_data,
        labels="inferred",
        label_mode="categorical",
        class_names=["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"],
        shuffle=False,
        seed=123,
        batch_size=batch_size,
        image_size=(32, 32),
)

# Initialize the base model using KerasCV's ResNet50
backbone = keras_cv.models.ResNet50Backbone.from_preset(
    input_shape=(32, 32, 3),
    preset = "resnet50_imagenet",
    load_weights=False,
)

# Create a new model that outputs the desired intermediate layer
base_model = tf.keras.Model(
    inputs=backbone.inputs,
    outputs=backbone.get_layer("v2_stack_0_block3_out").output
)

# Define the input tensor
inputs = tf.keras.Input(shape=(32, 32, 3))

# Pass the preprocessed input through the base model
x = base_model(inputs)

# Add global average pooling
x = tf.keras.layers.GlobalAveragePooling2D()(x)

# Add a dense layer for classification (assuming 10 classes)
x = tf.keras.layers.Dense(10)(x)

# Define the final model
model = tf.keras.Model(inputs, x)

# Compile the model with appropriate optimizer, loss, and metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

# Display the model's architecture
model.summary()
    
loss_0, acc_0 = model.evaluate(valid)
print(f"loss {loss_0}, acc {acc_0}")

checkpoint = tf.keras.callbacks.ModelCheckpoint(
        "best_model.weights.h5",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=3, verbose=1, min_lr=1e-7)

history = model.fit(
        train,
        validation_data=valid,
        epochs=75,
        callbacks=[checkpoint, lr_scheduler],
)

model.load_weights("best_model.weights.h5")

loss, acc = model.evaluate(valid)
print(f"final loss {loss}, final acc {acc}")

test_loss, test_acc = model.evaluate(test)
print(f"test loss {test_loss}, test acc {test_acc}")

### DO NOT MODIFY ABOVE THIS LINE, THIS IS THE FIXED MODEL ###

In [ ]:
model.evaluate(test)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    directory+"testing_data",
    shuffle = False,
    image_size=(32, 32),
    batch_size=1)

prob = model.predict(test_dataset)
predictions = []
for i in range(0, prob.shape[0]):
    predictions.append(np.argmax(prob[i,:])+1)


import pandas as pd

paths = test_dataset.file_paths

Ids = []
for x in paths:
    Ids.append(x.split("/")[-1])
    
df = pd.DataFrame()
df["Id"] = Ids
df["Predicted"] = predictions
df.to_csv("submission.csv", index=False)

from sklearn.metrics import classification_report
predictedlabel = pd.read_csv('/kaggle/working/submission.csv')
print(classification_report(thetruelabel['label'], predictedlabel['Predicted']))

In [ ]:
thetruelabel = pd.read_csv('/kaggle/input/truelabel/mapping_test (1).csv')